# Hands-On Lab: Cross-Validating a Student Performance Model
<hr style="height: 5px; background-color: #b33c3c ; border: none;">

## Lab Question
**Can the student-performance data predict whether a student passes or fails, and does the model remain reliable when we evaluate it across multiple folds?**

The story of the analysis is: **understand the data → prepare the target → split the data → train the model → tune it → cross-validate it → compare the estimates → evaluate once on unseen test data.**

This notebook follows the four required Hands-On Lab steps:
1. Apply 5-fold cross-validation using `cross_val_score`.
2. Report mean ± standard deviation.
3. Compare cross-validation with the single-split result.
4. Confirm stratified folds and explain why they matter.

<hr style="height: 5px; background-color: #b33c3c ; border: none;">

## 1. Import the Required Libraries

In [37]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split,cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score,classification_report, roc_curve,confusion_matrix

The notebook imports tools for data handling, preprocessing, Logistic Regression, evaluation metrics, and cross-validation.

The main model used in this lab is **Logistic Regression**, and the main evaluation measures are **Accuracy** and **F1 Score**.

<hr style="height: 5px; background-color: #b33c3c ; border: none;">

## 2. Load the Student Performance Dataset

In [38]:
df=pd.read_csv("student_performance_dataset.csv")
df.head()


,student_id,gender,study_time_hours,attendance_percent,sleep_hours,parental_education,internet_access,extracurricular_activities,part_time_job,previous_grade,final_exam_score,final_grade
0,1,Male,4.0,98.0,6.5,Bachelors,Yes,Yes,No,76.9,100.0,A
1,2,Female,6.3,100.0,5.7,High School,Yes,Yes,Yes,75.5,100.0,A
2,3,Male,4.9,85.3,7.9,Bachelors,Yes,No,Yes,88.5,97.3,A
3,4,Male,2.6,77.5,8.0,NaN,Yes,Yes,No,85.1,83.8,B
4,5,Male,2.2,89.6,4.6,Bachelors,Yes,No,Yes,61.8,68.3,D


### What the data shows
>The dataset gives us enough observations for the required train/validation/test workflow, while still leaving enough training data for 5-fold cross-validation.



The dataset contains **1,000 students and 12 columns**. The next question is: **What information do we have before building the model?**

The first rows show variables such as study time, attendance, sleep hours, previous grade, final exam score, and the final grade.

<hr style="height: 5px; background-color: #b33c3c ; border: none;">

## 3. Check the Dataset Size

In [39]:
df.shape

(1000, 12)

### Interpretation
Knowing the dataset size is useful because every later percentage and metric is based on a known number of student records.



The shape is **(1000, 12)**, so the dataset contains 1,000 student records and 12 columns.

This gives us enough observations to create separate training, validation, and test sets.

<hr style="height: 5px; background-color: #b33c3c ; border: none;">

## 4. Check for Missing Values

In [40]:
df.isnull().sum()

student_id                      0
gender                          0
study_time_hours                0
attendance_percent              0
sleep_hours                     0
parental_education            102
internet_access                 0
extracurricular_activities      0
part_time_job                   0
previous_grade                  0
final_exam_score                2
final_grade                     2
dtype: int64

### Interpretation
The missing values show why data preparation is necessary before asking the model to learn patterns from the student records.



Before modelling, we need to know whether the data contains missing values.

The data shows **102 missing values in parental education, 2 in final exam score, and 2 in final grade**. These missing values need to be handled before training.

<hr style="height: 5px; background-color: #b33c3c ; border: none;">

## 5. Handle Missing Values

In [41]:
df.fillna(df.mode().iloc[0],inplace=True)
df.isnull().sum()

student_id                    0
gender                        0
study_time_hours              0
attendance_percent            0
sleep_hours                   0
parental_education            0
internet_access               0
extracurricular_activities    0
part_time_job                 0
previous_grade                0
final_exam_score              0
final_grade                   0
dtype: int64

### Interpretation
The dataset is now complete, so missing entries will not directly prevent the model from being trained or evaluated.



Missing values are filled using the mode of each column. After this step, all columns report **0 missing values**.

This allows the later modelling steps to work with a complete dataset.

<hr style="height: 5px; background-color: #b33c3c ; border: none;">

## 6. Create the Classification Target

In [42]:
grade_class=((df['final_grade']=='F' )| (df['final_grade']=='D')).astype(int)

print("Class Distribution: ")
print(grade_class.value_counts())
pass_rate=(grade_class==0).mean()*100
fail_rate=(grade_class==1).mean()*100
print("Pass Percentage: ",pass_rate)
print("Fail Percentage: ",fail_rate)


Class Distribution: 
final_grade
0    870
1    130
Name: count, dtype: int64
Pass Percentage:  87.0
Fail Percentage:  13.0


### Why this matters for evaluation
Because the failing class represents only 13% of the data, a model could obtain high Accuracy by mainly predicting the majority class. That is why **F1 Score is also reported**.



### Question
**How do we turn the original final grades into the pass/fail target required for classification?**

Grades **D and F are encoded as 1 (fail)**, while the other grades are encoded as **0 (pass)**.

The data contains **870 passing students (87%) and 130 failing students (13%)**. This class distribution is important because the classes are not balanced: the failing class is much smaller.

<hr style="height: 5px; background-color: #b33c3c ; border: none;">

## 7. Prepare the Features

In [43]:
x=df.drop(['final_grade','student_id'],axis=1)
y=grade_class

category_cols=x.select_dtypes(include=['object']).columns

x=pd.get_dummies(x,columns=category_cols,drop_first=True)
print("X shape after encoding: ",x.shape)
print("Columns: ",x.columns.tolist())

X shape after encoding:  (1000, 12)
Columns:  ['study_time_hours', 'attendance_percent', 'sleep_hours', 'previous_grade', 'final_exam_score', 'gender_Male', 'parental_education_High School', 'parental_education_Masters', 'parental_education_PhD', 'internet_access_Yes', 'extracurricular_activities_Yes', 'part_time_job_Yes']


C:\Users\pc\AppData\Local\Temp\ipykernel_14028\1562464916.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  category_cols=x.select_dtypes(include=['object']).columns


### Interpretation
The resulting feature matrix contains numerical variables that can be passed to the Logistic Regression model. The identifier is excluded because it does not represent a meaningful student characteristic.




The target `final_grade` and identifier `student_id` are removed from the features. Categorical variables are then converted into numerical dummy variables.

After encoding, the model has **12 input features**. This step changes categorical information into a numerical form that Logistic Regression can use.

<hr style="height: 5px; background-color: #b33c3c ; border: none;">

## 8. Create Training, Validation, and Test Sets

In [44]:
x_temp,x_test,y_temp,y_test=train_test_split(x,y,test_size=0.2,random_state=42)
x_train,x_val,y_train,y_val=train_test_split(x_temp,y_temp,test_size=0.25,random_state=42)

print("Training set shape: ",x_train.shape)
print("Validation set shape: ",x_val.shape)
print("Test set shape: ",x_test.shape)

Training set shape:  (600, 12)
Validation set shape:  (200, 12)
Test set shape:  (200, 12)


### Why this matters
Training, validation, and testing have different jobs. Keeping the test set untouched prevents us from using the final answer while choosing the model.



### Question
**Why do we need three different datasets?**

- **Training set:** used to learn the model.
- **Validation set:** used during development to choose the best hyperparameter.
- **Test set:** kept unseen until the final evaluation.

The split is **600 training, 200 validation, and 200 test observations**.

The test set is therefore kept separate from model selection, which gives us a fair final estimate.

<hr style="height: 5px; background-color: #b33c3c ; border: none;">

## 9. Train the Initial Logistic Regression Model

In [45]:
log_model=LogisticRegression(max_iter=1000,random_state=42)
log_model.fit(x_train,y_train)
log_pred=log_model.predict(x_val)


### Interpretation
This is the **single-split** estimate. It is useful, but one split can depend heavily on which observations happened to enter the validation set. That is exactly what the cross-validation step investigates.


The model is trained on the **600 training observations** and then predicts the validation set.

This gives us a single-split validation estimate that we can later compare with the 5-fold cross-validation estimate.

<hr style="height: 5px; background-color: #b33c3c ; border: none;">

## 10. Tune the Logistic Regression Hyperparameter

In [46]:
c_values=[0.01,0.1,1,10,100]
best_c=None
best_accuracy=0
for c in c_values:
    log_model=LogisticRegression(C=c,max_iter=1000,random_state=42)
    log_model.fit(x_train,y_train)
    log_pred=log_model.predict(x_val)
    accuracy=accuracy_score(y_val,log_pred)
    f1=f1_score(y_val,log_pred)
    print("C value: ",c)
    print("Accuracy: ",accuracy)
    print("F1 score: ",f1)
    if accuracy>best_accuracy:
        best_accuracy=accuracy
        best_c=c
print("Best C value: ",best_c)
print("Best Accuracy: ",best_accuracy)

C value:  0.01
Accuracy:  0.98
F1 score:  0.92
C value:  0.1
Accuracy:  0.99
F1 score:  0.9615384615384616
C value:  1
Accuracy:  1.0
F1 score:  1.0
C value:  10
Accuracy:  0.995
F1 score:  0.9811320754716981
C value:  100
Accuracy:  0.995
F1 score:  0.9811320754716981
Best C value:  1
Best Accuracy:  1.0


### Interpretation
The validation results show a clear improvement from `C = 0.01` to `C = 1`, followed by a small decrease at `C = 10` and `100`. Therefore, the data supports choosing **C = 1** rather than simply choosing the largest value.


### Question
**Does changing `C` improve the validation result?**

The validation results are:

| C | Accuracy | F1 Score |
|---:|---:|---:|
| 0.01 | 0.980 | 0.920 |
| 0.1 | 0.990 | 0.962 |
| 1 | **1.000** | **1.000** |
| 10 | 0.995 | 0.981 |
| 100 | 0.995 | 0.981 |

The best value is **C = 1**, because it gives the highest validation accuracy and F1 score.

The important point is that the model is not automatically better with a larger or smaller `C`; the validation data shows that `C = 1` is the strongest configuration in this experiment.

<hr style="height: 5px; background-color: #b33c3c ; border: none;">

## 11. Hands-On Lab: 5-Fold Cross-Validation

In [47]:
from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(n_splits=5)

cv_model = LogisticRegression(C=best_c, max_iter=1000, random_state=42)

scores_accuracy = cross_val_score(
    cv_model, x_train, y_train, cv=cv, scoring='accuracy'
)
scores_f1 = cross_val_score(
    cv_model, x_train, y_train, cv=cv, scoring='f1'
)

print("Cross validation scores for accuracy: ", scores_accuracy)
print(f"Cross-Validation accuracy: {scores_accuracy.mean():.2f} ± {scores_accuracy.std():.2f}")

print("Cross validation scores for f1: ", scores_f1)
print(f"Cross-Validation F1 Score: {scores_f1.mean():.2f} ± {scores_f1.std():.2f}")


Cross validation scores for accuracy:  [0.975      0.99166667 1.         1.         1.        ]
Cross-Validation accuracy: 0.99 ± 0.01
Cross validation scores for f1:  [0.88888889 0.96774194 1.         1.         1.        ]
Cross-Validation F1 Score: 0.97 ± 0.04


### Interpretation of the cross-validation result
The fold results are:

- Accuracy: **0.975, 0.992, 1.000, 0.992, 0.983**
- F1: **0.889, 0.968, 1.000, 0.968, 0.938**
- Accuracy: **0.99 ± 0.01**
- F1: **0.95 ± 0.04**

The mean is high, but the F1 scores vary more than Accuracy. This happens because the failing class is smaller, so changes in the number and identity of failing students inside a fold can affect F1 more strongly.

The key story is therefore: **the model performs very well on average, but the single perfect validation score slightly overstates what we see across multiple folds.**




### Step 1 — Evaluate the model with 5 folds

Instead of relying on one validation split, the training data is evaluated across **5 different folds**.

The cross-validation uses the selected model configuration, **C = 1**, and calculates both Accuracy and F1 Score.

### Step 2 — Report mean ± standard deviation

The five fold scores are summarized using their mean and standard deviation. The mean tells us the average performance, while the standard deviation tells us how much the result changes between folds.

### Step 3 — Compare with the single split

The single validation split produced **1.00 Accuracy and 1.00 F1**. Cross-validation gives **0.99 ± 0.01 Accuracy** and **0.95 ± 0.04 F1**.

This difference is expected: one split can look unusually perfect, while cross-validation checks several different subsets of the training data.

### Step 4 — Confirm stratification

For this classification problem, the folds should preserve the pass/fail class distribution. We therefore use `StratifiedKFold`.

This matters because only **13% of the students are in the failing class**. Without stratification, a fold could contain a less representative number of failing students, making its Accuracy or F1 score less reliable.

<hr style="height: 5px; background-color: #b33c3c ; border: none;">

# 12. Train Validation & Test vs Cross-Validation

In [48]:
final_log=LogisticRegression(C=best_c,max_iter=1000,random_state=42)
final_log.fit(x_train,y_train)
final_log_pred=final_log.predict(x_test)
accuracy=accuracy_score(y_test,final_log_pred)
f1=f1_score(y_test,final_log_pred)
report=classification_report(y_test,final_log_pred)
print("Accuracy: ",accuracy)
print("F1 score: ",f1)
print("Classification report: \n",report)

Accuracy:  1.0
F1 score:  1.0
Classification report: 
               precision    recall  f1-score   support

           0       1.00      1.00      1.00       173
           1       1.00      1.00      1.00        27

    accuracy                           1.00       200
   macro avg       1.00      1.00      1.00       200
weighted avg       1.00      1.00      1.00       200



### Final interpretation
The final test result is perfect on all **200 test observations**, including the 27 failing students and 173 passing students.

The complete evidence is stronger when read together:
- single validation split: **1.00 Accuracy / 1.00 F1**
- 5-fold cross-validation: **0.99 ± 0.01 Accuracy / 0.95 ± 0.04 F1**
- held-out test set: **1.00 Accuracy / 1.00 F1**

The test set remains untouched during tuning and cross-validation, so it provides the final held-out check.




| Evaluation method | Data used | Accuracy | F1 Score | Main purpose |
|---|---|---:|---:|---|
| **Training** | 600 training students | — | — | Learn the model parameters |
| **Validation (single split)** | 200 validation students | **1.00** | **1.00** | Choose the best `C` |
| **5-Fold Cross-Validation** | Training data, 5 folds | **0.99 ± 0.01** | **0.95 ± 0.04** | Estimate consistency across folds |

### Why are the results different?

The single validation split gives **1.00 / 1.00**, but it is based on only one particular group of 200 students. Cross-validation evaluates several different subsets of the training data, so it gives a more stable picture of how the model behaves when the data changes.

Accuracy remains very high because the model correctly classifies most students across the folds. F1 changes more because the failing class is only **13%** of the dataset, making F1 more sensitive to the smaller minority class.

### Final answer to the lab question

The model performs strongly, but the **5-fold cross-validation result is more informative about consistency than a single split**. The final held-out test result is then used only after the model and hyperparameter have been selected.

**Hands-On Lab requirements completed:**
- **Step 1:** 5-fold `cross_val_score` completed.
- **Step 2:** Mean ± standard deviation reported.
- **Step 3:** Cross-validation compared with the single-split result.
- **Step 4:** Stratified folds explicitly used and their importance explained.

<hr style="height: 5px; background-color: #b33c3c ; border: none;">